In [23]:
import glob
import os
import re
import pandas as pd

data_dir = os.path.join("..", "data")

# Panel year ("Anno") = population reference year (1 January of that year).
# Income of tax year T is attached to panel year T + INCOME_LAG (income 2020 -> Anno 2021, ..., 2024 -> 2025).
INCOME_LAG = 1

# ---------- ISTAT population: one file per province per reference year ----------
def load_posas(path):
    """Read a POSAS csv; finds the header row instead of assuming skiprows=1."""
    with open(path, encoding="utf-8-sig") as f:
        skip = next(i for i, line in enumerate(f) if "Codice comune" in line)
    df = pd.read_csv(path, sep=";", skiprows=skip, encoding="utf-8-sig",
                     dtype={"Codice comune": str})
    df = df[df["Codice comune"].str.fullmatch(r"\d{6}", na=False)].copy()
    df["Anno"] = int(re.search(r"POSAS_(\d{4})_", os.path.basename(path)).group(1))
    return df

posas_files = sorted(glob.glob(os.path.join(data_dir, "POSAS_*_it_05[45]_*.csv")))
umbria_demographics = pd.concat([load_posas(p) for p in posas_files], ignore_index=True)
umbria_demographics["Codice_Comune"] = umbria_demographics["Codice comune"].astype(int)

# Prendi le chiavi dei 92 comuni sicuri dell'Umbria
comuni_keys = umbria_demographics["Codice_Comune"].unique()


# ---------- MEF income: explicit loading for all 5 years (2020-2024) ----------
income_filenames = [
    "Redditi_e_principali_variabili_IRPEF_su_base_comunale_CSV_2020.csv",
    "Redditi_e_principali_variabili_IRPEF_su_base_comunale_CSV_2021.csv",
    "Redditi_e_principali_variabili_IRPEF_su_base_comunale_CSV_2022.csv",
    "Redditi_e_principali_variabili_IRPEF_su_base_comunale_CSV_2023.csv",
    "Redditi_e_principali_variabili_IRPEF_su_base_comunale_CSV_2024.csv"
]

print(income_filenames)

income_frames = []
for fname in income_filenames:
    path = os.path.join(data_dir, fname)
    if os.path.exists(path):
        df = pd.read_csv(path, sep=";", encoding="utf-8-sig", dtype=str,
                         index_col=False, usecols=lambda column: True)
        
        # Filtro sicuro basato sulla provincia (PG e TR)
        if "Sigla Provincia" in df.columns:
            df_umbria = df[df["Sigla Provincia"].isin(["PG", "TR"])].copy()
        else:
            # Fallback sul codice istat se la sigla provincia mancasse
            df["Codice Istat Comune"] = pd.to_numeric(df["Codice Istat Comune"], errors="coerce")
            df_umbria = df[df["Codice Istat Comune"].between(54000, 55999)].copy()
            
        income_frames.append(df_umbria)
    else:
        print(f"Attenzione: file non trovato -> {fname}")

umbria_income = pd.concat(income_frames, ignore_index=True)

# Pulizia finale e conversione codici
umbria_income["Codice Istat Comune"] = umbria_income["Codice Istat Comune"].astype(str).str.strip()
umbria_income["Codice_Comune"] = pd.to_numeric(umbria_income["Codice Istat Comune"], errors="coerce").astype(int)
umbria_income["Anno di imposta"] = pd.to_numeric(umbria_income["Anno di imposta"])
umbria_income["Anno"] = umbria_income["Anno di imposta"] + INCOME_LAG

# ---------- Verifiche finali ----------
print("Population files count:", len(posas_files))
print("Population rows per year:\n", umbria_demographics.groupby("Anno")["Codice_Comune"].nunique().to_string())
print("\nIncome rows per tax year:\n", umbria_income.groupby("Anno di imposta")["Codice_Comune"].nunique().to_string())

['Redditi_e_principali_variabili_IRPEF_su_base_comunale_CSV_2020.csv', 'Redditi_e_principali_variabili_IRPEF_su_base_comunale_CSV_2021.csv', 'Redditi_e_principali_variabili_IRPEF_su_base_comunale_CSV_2022.csv', 'Redditi_e_principali_variabili_IRPEF_su_base_comunale_CSV_2023.csv', 'Redditi_e_principali_variabili_IRPEF_su_base_comunale_CSV_2024.csv']
Population files count: 12
Population rows per year:
 Anno
2021    92
2022    92
2023    92
2024    92
2025    92
2026    92

Income rows per tax year:
 Anno di imposta
2020    92
2021    92
2022    92
2023    92
2024    92


In [16]:
import os
import pandas as pd

data_dir = os.path.join("..", "data")
file_2020 = os.path.join(data_dir, "Redditi_e_principali_variabili_IRPEF_su_base_comunale_CSV_2020.csv")

# Read just the first row to see how the Ministry named the columns in 2020
df_test = pd.read_csv(file_2020, sep=";", nrows=1, encoding="utf-8-sig")

print("Colonne in 2020:")
print(df_test.columns.tolist())

Colonne in 2020:
['Anno di imposta', 'Codice catastale', 'Codice Istat Comune', 'Denominazione Comune', 'Sigla Provincia', 'Regione', 'Codice Istat Regione', 'Numero contribuenti', 'Reddito da fabbricati - Frequenza', 'Reddito da fabbricati - Ammontare in euro', 'Reddito da lavoro dipendente e assimilati - Frequenza', 'Reddito da lavoro dipendente e assimilati - Ammontare in euro', 'Reddito da pensione - Frequenza', 'Reddito da pensione - Ammontare in euro', 'Reddito da lavoro autonomo (comprensivo dei valori nulli) - Frequenza', 'Reddito da lavoro autonomo (comprensivo dei valori nulli) - Ammontare in euro', "Reddito di spettanza dell'imprenditore in contabilita' ordinaria  (comprensivo dei valori nulli) - Frequenza", "Reddito di spettanza dell'imprenditore in contabilita' ordinaria  (comprensivo dei valori nulli) - Ammontare in euro", "Reddito di spettanza dell'imprenditore in contabilita' semplificata (comprensivo dei valori nulli) - Frequenza", "Reddito di spettanza dell'imprendito

In [24]:
# Standardize names of comuni (still needed for the OMI macro-area mapping below)
umbria_income['Comune_Key'] = umbria_income['Denominazione Comune'].astype(str).str.strip().str.upper()

# Keep only realistic ages (drops the 999 "total" row), then sum ages -> population per comune AND year
umbria_demographics['Età_Num'] = pd.to_numeric(umbria_demographics['Età'], errors='coerce')
umbria_filtered = umbria_demographics[umbria_demographics['Età_Num'] < 150]

istat_aggreg = (
    umbria_filtered.groupby(['Codice_Comune', 'Anno'], as_index=False)['Totale'].sum()
    .rename(columns={'Totale': 'Popolazione Totale'})
)

print("Anni disponibili in umbria_income:", sorted(umbria_income['Anno'].unique()))
print("Anni disponibili in istat_aggreg:", sorted(istat_aggreg['Anno'].unique()))

# Merge on ISTAT code + panel year (NOT on the comune name: "CITTÀ" vs "CITTA'" silently dropped 2 comuni)
dataset_completo = pd.merge(
    umbria_income, istat_aggreg,
    on=['Codice_Comune', 'Anno'], how='inner', validate='one_to_one'
)
dataset_completo = dataset_completo.sort_values(['Codice_Comune', 'Anno']).reset_index(drop=True)

# Verify merge: which years were dropped, and 92 comuni per remaining year
print("Anni popolazione scartati:", sorted(set(istat_aggreg['Anno']) - set(dataset_completo['Anno'])))
print("Anni reddito scartati:   ", sorted(set(umbria_income['Anno']) - set(dataset_completo['Anno'])))
print(f"Righe totali dopo il merge: {len(dataset_completo)}")
print(dataset_completo.groupby('Anno').size().to_string())

selected_row = dataset_completo[dataset_completo['Comune_Key'] == 'ASSISI']
display(selected_row[['Comune_Key', 'Anno', 'Popolazione Totale', 'Numero contribuenti']])

Anni disponibili in umbria_income: [np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Anni disponibili in istat_aggreg: [np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]
Anni popolazione scartati: [2026]
Anni reddito scartati:    []
Righe totali dopo il merge: 460
Anno
2021    92
2022    92
2023    92
2024    92
2025    92


,Comune_Key,Anno,Popolazione Totale,Numero contribuenti
0,ASSISI,2021,28004.0,20002
1,ASSISI,2022,27880.0,20738
2,ASSISI,2023,27671.0,20908
3,ASSISI,2024,27507.0,21192
4,ASSISI,2025,27468.0,20543


**Feature Distinction: Total Population vs. Taxpayers**
When comparing the demographic and economic features, there is a consistent gap between `Popolazione Totale` and `Numero contribuenti`. The total population counts every registered resident in the municipality. The taxpayer metric only accounts for individuals who filed an income tax return (IRPEF). This structural gap filters out children, students, and adults without independent declarable income, making the taxpayer count a useful proxy for the active workforce and income-generating base of the town.

In [25]:
#From OMI Data, prices in €/m² for each macro-area (from OMI tables 11 and 15)
# OMI prices in €/m² per macro-area and year (tables 11 and 15 of each OMI report).
omi_2025 = {
    # Perugia Macro-areas (Table 11)
    'ALTO TEVERE': 782,
    'ASSISIATE': 982,
    'COLLI DEL TRASIMENO': 802,
    'EUGUBINO-GUALDESE': 705,
    'FOLIGNATE': 886,
    'MEDIA VALLE DEL TEVERE': 825,
    'MONTI MARTANI': 556,
    'PERUGINO': 940,
    'SPOLETINO': 958,
    'PERUGIA': 1075, # Solo capoluogo
    
    # Terni Macro-areas (Table 15)
    'AMERINO': 782,
    'NARNESE': 832,
    'ORVIETANO': 898,
    'VAL NERINA': 743,
    'TERNI': 895 # Solo capoluogo
}

omi_prices_by_year = {
    2021: {
         # Perugia Macro-areas (Table 11)
            'ALTO TEVERE': 799,
            'ASSISIATE': 993,
            'COLLI DEL TRASIMENO': 811,
            'EUGUBINO-GUALDESE': 715,
            'FOLIGNATE': 910,
            'MEDIA VALLE DEL TEVERE': 855,
            'MONTI MARTANI': 558,
            'PERUGINO': 927,
            'SPOLETINO': 1002,
            'PERUGIA': 1148, # Solo capoluogo
            
            # Terni Macro-areas (Table 15)
            'AMERINO': 807,
            'NARNESE': 882,
            'ORVIETANO': 1026,
            'VAL NERINA': 761,
            'TERNI': 983 # Solo capoluogo

    },   
    2022: {
         # Perugia Macro-areas (Table 11)
            'ALTO TEVERE': 800,
            'ASSISIATE': 996,
            'COLLI DEL TRASIMENO': 811,
            'EUGUBINO-GUALDESE': 712,
            'FOLIGNATE': 905,
            'MEDIA VALLE DEL TEVERE': 848,
            'MONTI MARTANI': 557,
            'PERUGINO': 929,
            'SPOLETINO': 991,
            'PERUGIA': 1119, # Solo capoluogo
            
            # Terni Macro-areas (Table 15)
            'AMERINO': 758,
            'NARNESE': 786,
            'ORVIETANO': 853,
            'VAL NERINA': 700,
            'TERNI': 787 # Solo capoluogo
    },   
    2023: {
         # Perugia Macro-areas (Table 11)
            'ALTO TEVERE': 790,
            'ASSISIATE': 992,
            'COLLI DEL TRASIMENO': 802,
            'EUGUBINO-GUALDESE': 705,
            'FOLIGNATE': 899,
            'MEDIA VALLE DEL TEVERE': 838,
            'MONTI MARTANI': 556,
            'PERUGINO': 926,
            'SPOLETINO': 973,
            'PERUGIA': 1099, # Solo capoluogo
            
            # Terni Macro-areas (Table 15)
            'AMERINO': 746,
            'NARNESE': 776,
            'ORVIETANO': 853,
            'VAL NERINA': 691,
            'TERNI': 786 # Solo capoluogo
    },   
    2024: {
         # Perugia Macro-areas (Table 11)
            'ALTO TEVERE': 786,
            'ASSISIATE': 987,
            'COLLI DEL TRASIMENO': 801,
            'EUGUBINO-GUALDESE': 711,
            'FOLIGNATE': 890,
            'MEDIA VALLE DEL TEVERE': 830,
            'MONTI MARTANI': 556,
            'PERUGINO': 938,
            'SPOLETINO': 966,
            'PERUGIA': 1078, # Solo capoluogo
            
            # Terni Macro-areas (Table 15)
            'AMERINO': 743,
            'NARNESE': 788,
            'ORVIETANO': 870,
            'VAL NERINA': 689,
            'TERNI': 809 # Solo capoluogo
    },   # TODO
    2025: omi_2025,
}

comune_to_macroarea = {
    # --- PROVINCIA DI PERUGIA ---
    # Alto Tevere
    'CITERNA': 'ALTO TEVERE', 'CITTA\' DI CASTELLO': 'ALTO TEVERE', 'LISCIANO NICCONE': 'ALTO TEVERE', 
    'MONTE SANTA MARIA TIBERINA': 'ALTO TEVERE', 'MONTONE': 'ALTO TEVERE', 'PIETRALUNGA': 'ALTO TEVERE', 
    'SAN GIUSTINO': 'ALTO TEVERE', 'UMBERTIDE': 'ALTO TEVERE',
    
    # Assisiate
    'ASSISI': 'ASSISIATE', 'BASTIA UMBRA': 'ASSISIATE', 'BETTONA': 'ASSISIATE', 'CANNARA': 'ASSISIATE',
    
    # Colli del Trasimeno
    'CASTIGLIONE DEL LAGO': 'COLLI DEL TRASIMENO', 'CITTA\' DELLA PIEVE': 'COLLI DEL TRASIMENO', 
    'MAGIONE': 'COLLI DEL TRASIMENO', 'PACIANO': 'COLLI DEL TRASIMENO', 'PANICALE': 'COLLI DEL TRASIMENO', 
    'PASSIGNANO SUL TRASIMENO': 'COLLI DEL TRASIMENO', 'PIEGARO': 'COLLI DEL TRASIMENO', 
    'TUORO SUL TRASIMENO': 'COLLI DEL TRASIMENO',
    
    # Eugubino-Gualdese
    'COSTACCIARO': 'EUGUBINO-GUALDESE', 'FOSSATO DI VICO': 'EUGUBINO-GUALDESE', 'GUALDO TADINO': 'EUGUBINO-GUALDESE', 
    'GUBBIO': 'EUGUBINO-GUALDESE', 'SCHEGGIA E PASCELUPO': 'EUGUBINO-GUALDESE', 'SIGILLO': 'EUGUBINO-GUALDESE', 
    'VALFABBRICA': 'EUGUBINO-GUALDESE',
    
    # Folignate
    'BEVAGNA': 'FOLIGNATE', 'FOLIGNO': 'FOLIGNATE', 'MONTEFALCO': 'FOLIGNATE', 'NOCERA UMBRA': 'FOLIGNATE', 
    'SPELLO': 'FOLIGNATE', 'TREVI': 'FOLIGNATE', 'VALTOPINA': 'FOLIGNATE',
    
    # Media Valle del Tevere
    'COLLAZZONE': 'MEDIA VALLE DEL TEVERE', 'DERUTA': 'MEDIA VALLE DEL TEVERE', 'FRATTA TODINA': 'MEDIA VALLE DEL TEVERE', 
    'MARSCIANO': 'MEDIA VALLE DEL TEVERE', 'MONTE CASTELLO DI VIBIO': 'MEDIA VALLE DEL TEVERE', 'TODI': 'MEDIA VALLE DEL TEVERE',
    
    # Monti Martani
    'CASTEL RITALDI': 'MONTI MARTANI', 'GIANO DELL\'UMBRIA': 'MONTI MARTANI', 'MASSA MARTANA': 'MONTI MARTANI',
    
    # Perugino
    'CORCIANO': 'PERUGINO', 'PERUGIA': 'PERUGIA', 'TORGIANO': 'PERUGINO',
    
    # Spoletino
    'CAMPELLO SUL CLITUNNO': 'SPOLETINO', 'GUALDO CATTANEO': 'SPOLETINO', 'SPOLETO': 'SPOLETINO',
    
    # Val Nerina (Perugia)
    'CASCIA': 'VAL NERINA', 'CERRETO DI SPOLETO': 'VAL NERINA', 'MONTELEONE DI SPOLETO': 'VAL NERINA', 
    'NORCIA': 'VAL NERINA', 'POGGIODOMO': 'VAL NERINA', 'PRECI': 'VAL NERINA', 'SANT\'ANATOLIA DI NARCO': 'VAL NERINA', 
    'SCHEGGINO': 'VAL NERINA', 'SELLANO': 'VAL NERINA', 'VALLO DI NERA': 'VAL NERINA',

    # --- PROVINCIA DI TERNI ---
    # Amerino
    'ACQUASPARTA': 'AMERINO', 'ALVIANO': 'AMERINO', 'AMELIA': 'AMERINO', 'ATTIGLIANO': 'AMERINO', 
    'AVIGLIANO UMBRO': 'AMERINO', 'GIOVE': 'AMERINO', 'GUARDEA': 'AMERINO', 'LUGNANO IN TEVERINA': 'AMERINO', 
    'MONTECASTRILLI': 'AMERINO', 'MONTECCHIO': 'AMERINO', 'PENNA IN TEVERINA': 'AMERINO',
    
    # Narnese
    'CALVI DELL\'UMBRIA': 'NARNESE', 'NARNI': 'NARNESE', 'OTRICOLI': 'NARNESE', 
    'SAN GEMINI': 'NARNESE', 'STRONCONE': 'NARNESE',
    
    # Orvietano
    'ALLERONA': 'ORVIETANO', 'BASCHI': 'ORVIETANO', 'CASTEL GIORGIO': 'ORVIETANO', 'CASTEL VISCARDO': 'ORVIETANO', 
    'FABRO': 'ORVIETANO', 'FICULLE': 'ORVIETANO', 'MONTEGABBIONE': 'ORVIETANO', 'MONTELEONE D\'ORVIETO': 'ORVIETANO', 
    'ORVIETO': 'ORVIETANO', 'PARRANO': 'ORVIETANO', 'PORANO': 'ORVIETANO', 'SAN VENANZO': 'ORVIETANO',
    
    # Terni Capoluogo
    'TERNI': 'TERNI',
    
    # Valnerina (Terni)
    'ARRONE': 'VAL NERINA', 'FERENTILLO': 'VAL NERINA', 'MONTEFRANCO': 'VAL NERINA', 'POLINO': 'VAL NERINA'
}

# Long table (Macroarea_OMI, Anno, Prezzo_Medio_Mq) and merge on macro-area + year
omi_long = (
    pd.DataFrame(omi_prices_by_year).rename_axis('Macroarea_OMI').reset_index()
    .melt(id_vars='Macroarea_OMI', var_name='Anno', value_name='Prezzo_Medio_Mq')
)
omi_long['Anno'] = omi_long['Anno'].astype(int)

dataset_completo['Macroarea_OMI'] = dataset_completo['Comune_Key'].map(comune_to_macroarea)
dataset_completo = (
    dataset_completo.drop(columns='Prezzo_Medio_Mq', errors='ignore')
    .merge(omi_long, on=['Macroarea_OMI', 'Anno'], how='left', validate='many_to_one')
)

display(dataset_completo[['Comune_Key', 'Anno', 'Macroarea_OMI', 'Prezzo_Medio_Mq']].head(10))

# Sanity check: macro-area/year combinations without a price (empty years above, or keys like 'VAL NERINA' missing)
no_price = (dataset_completo[dataset_completo['Prezzo_Medio_Mq'].isna()]
            .groupby(['Anno', 'Macroarea_OMI']).size())
print("Righe senza prezzo OMI:")
print(no_price.to_string() if len(no_price) else "none")

,Comune_Key,Anno,Macroarea_OMI,Prezzo_Medio_Mq
0,ASSISI,2021,ASSISIATE,993
1,ASSISI,2022,ASSISIATE,996
2,ASSISI,2023,ASSISIATE,992
3,ASSISI,2024,ASSISIATE,987
4,ASSISI,2025,ASSISIATE,982
5,BASTIA UMBRA,2021,ASSISIATE,993
6,BASTIA UMBRA,2022,ASSISIATE,996
7,BASTIA UMBRA,2023,ASSISIATE,992
8,BASTIA UMBRA,2024,ASSISIATE,987
9,BASTIA UMBRA,2025,ASSISIATE,982


Righe senza prezzo OMI:
none


In [26]:
from sklearn.impute import SimpleImputer

# 1. Identify the pre-calculated MEF Total Income column
income_col = [c for c in dataset_completo.columns if 'Reddito complessivo' in c and 'Ammontare' in c]

if not income_col:
    print("Colonne di ammontare disponibili nel dataset MEF:")
    print([c for c in dataset_completo.columns if 'Ammontare' in c])
else:
    col_reddito = 'Reddito complessivo - Ammontare in euro'
    col_reddito = col_reddito if col_reddito in income_col else income_col[0]
    print(f"Colonna utilizzata per il reddito totale: {col_reddito}")

    # 2. Convert to numeric (forces any empty cells or text into NaN)
    core_cols = [col_reddito, 'Popolazione Totale', 'Numero contribuenti']
    for c in core_cols:
        dataset_completo[c] = pd.to_numeric(dataset_completo[c], errors='coerce')

    # 3. Impute NaNs with the median *within each year* (a pooled median would mix 2021 and 2024 values)
    for anno, idx in dataset_completo.groupby('Anno').groups.items():
        dataset_completo.loc[idx, core_cols] = SimpleImputer(strategy="median").fit_transform(
            dataset_completo.loc[idx, core_cols]
        )

    # 4. Pro capita metrics
    dataset_completo['Reddito_Pro_Capite'] = dataset_completo[col_reddito] / dataset_completo['Popolazione Totale']
    dataset_completo['Reddito_Medio_Contribuente'] = dataset_completo[col_reddito] / dataset_completo['Numero contribuenti']

    # 5. Year-over-year change per comune (first year of each comune is NaN by construction)
    dataset_completo = dataset_completo.sort_values(['Codice_Comune', 'Anno']).reset_index(drop=True)
    assert (dataset_completo.groupby('Codice_Comune')['Anno'].diff().dropna() == 1).all(), \
        "Gap in the years for some comune: diff/pct_change would span more than one year"

    yoy_cols = ['Popolazione Totale', 'Numero contribuenti', col_reddito,
                'Reddito_Pro_Capite', 'Reddito_Medio_Contribuente', 'Prezzo_Medio_Mq']
    grp = dataset_completo.groupby('Codice_Comune')
    for c in yoy_cols:
        dataset_completo[f'{c}_diff'] = grp[c].diff()                          # absolute change
        dataset_completo[f'{c}_yoy_pct'] = grp[c].pct_change(fill_method=None) * 100   # % change

    # 6. Display
    cols_view = ['Comune_Key', 'Anno', 'Popolazione Totale', 'Popolazione Totale_yoy_pct',
                 'Reddito_Pro_Capite', 'Reddito_Pro_Capite_yoy_pct',
                 'Prezzo_Medio_Mq', 'Prezzo_Medio_Mq_yoy_pct']
    display(dataset_completo[cols_view].head(12))

Colonna utilizzata per il reddito totale: Reddito complessivo - Ammontare in euro


/home/leonardo/Desktop/Projects/umbria-housing-prices/.pixi/envs/default/lib/python3.14/site-packages/sklearn/impute/_base.py:647: UserWarning: Skipping features without any observed values: ['Reddito complessivo - Ammontare in euro']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


ValueError: Must have equal len keys and value when setting with an ndarray